# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [ ]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform
import sys
from pathlib import Path

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !{sys.executable} -m pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!{sys.executable} -m pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

repo_root = Path.cwd().resolve().parent
assert (repo_root / "microwakeword").exists(), f"Expected microwakeword package at {repo_root}"
!{sys.executable} -m pip install -e "{repo_root}"

In [ ]:
# Generates 1 sample of the target word for manual verification using Qwen VoiceDesign.
# NOTE: qwen-tts is heavy. If dependency conflicts appear, run this notebook with a dedicated
# Python env for synthesis (e.g. .venv_qwen) and keep training in your existing .venv.

target_word = '넙죽아'
qwen_model_id = 'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign'
language = 'Korean'

import sys
import subprocess
from pathlib import Path
from IPython.display import Audio

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('pyproject.toml not found from current notebook location')

project_root = find_project_root(Path.cwd())
generated_samples_dir = Path('generated_samples').resolve()
generated_samples_dir.mkdir(parents=True, exist_ok=True)

print(f'project_root={project_root}')
print(f'generated_samples_dir={generated_samples_dir.resolve()}')
print('Installing/validating qwen-tts runtime...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'qwen-tts', 'soundfile', 'scipy'], check=False)

cmd = [
    sys.executable, '-m', 'nubjuk_wakeword.cli', 'synth',
    '--target-word', target_word,
    '--model-id', qwen_model_id,
    '--language', language,
    '--max-samples', '1',
    '--batch-size', '1',
    '--output-dir', str(generated_samples_dir),
    '--qc-manifest', str(generated_samples_dir / 'qwen_qc_manifest.csv'),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

generated = sorted(generated_samples_dir.glob('*.wav'))
assert generated, 'No generated sample wav file found in generated_samples/'
Audio(filename=str(generated[0]), autoplay=True)


In [ ]:
# Generates a larger amount of wake word samples with Korean text input.
# Quality gate runs automatically inside cli synth and writes qwen_qc_manifest.csv.

import sys
import subprocess
import numpy as np
from pathlib import Path
from scipy.io import wavfile

max_samples = 1000
batch_size = 8

instructs = [
    '차분한 한국어 여성 목소리, 또렷한 발음',
    '부드러운 한국어 남성 목소리, 중간 속도',
    '밝고 경쾌한 톤, 자연스러운 한국어 발화',
    '조용한 환경에서 또렷하게 말하는 톤',
]

cmd = [
    sys.executable, '-m', 'nubjuk_wakeword.cli', 'synth',
    '--target-word', target_word,
    '--model-id', qwen_model_id,
    '--language', language,
    '--max-samples', str(max_samples),
    '--batch-size', str(batch_size),
    '--output-dir', str(generated_samples_dir),
    '--qc-manifest', str(generated_samples_dir / 'qwen_qc_manifest.csv'),
]
for instr in instructs:
    cmd.extend(['--instruct', instr])

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

generated = sorted(generated_samples_dir.glob('*.wav'))
print(f'Generated samples (after QC): {len(generated)}')
assert len(generated) >= 100, 'Too few generated samples. Adjust VoiceDesign prompts / max_samples.'

probe = generated[: min(20, len(generated))]
rms_values = []
duration_values = []
for wav_path in probe:
    sr, pcm = wavfile.read(wav_path)
    if pcm.ndim > 1:
        pcm = pcm[:, 0]
    pcm = pcm.astype(np.float32)
    rms_values.append(float(np.sqrt(np.mean(np.square(pcm)))))
    duration_values.append(len(pcm) / float(sr))

print(f'Probe RMS min/median/max: {min(rms_values):.1f} / {np.median(rms_values):.1f} / {max(rms_values):.1f}')
print(f'Probe duration(s) min/median/max: {min(duration_values):.3f} / {np.median(duration_values):.3f} / {max(duration_values):.3f}')
qc_manifest = generated_samples_dir / 'qwen_qc_manifest.csv'
print(f'QC manifest: {qc_manifest.resolve()} (exists={qc_manifest.exists()})')


In [ ]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.


import datasets
import scipy
import os
import tarfile
import zipfile
import urllib.request

import numpy as np

from pathlib import Path
from tqdm import tqdm

## Download MIR RIR data

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
download_ok = True
if not os.path.exists(out_dir):
    try:
        urllib.request.urlretrieve(link, out_dir)
    except Exception as exc:
        download_ok = False
        print(f"Warning: failed to download AudioSet archive: {exc}")

if download_ok and not os.path.exists("audioset/audio"):
    with tarfile.open(out_dir) as tf:
        tf.extractall("audioset")

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

if download_ok and os.path.exists("audioset/audio"):
    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fname = "fma_xs.zip"
link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
out_dir = f"fma/{fname}"
if not os.path.exists(out_dir):
    urllib.request.urlretrieve(link, out_dir)
if not os.path.exists("fma/fma_small"):
    with zipfile.ZipFile(out_dir) as zf:
        zf.extractall("fma")

output_dir = "./fma_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# Save clips to 16-bit PCM wav files
fma_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(fma_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


In [ ]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

def dirs_with_audio(paths):
    valid = []
    for p in paths:
        pp = Path(p)
        if not pp.exists():
            continue
        has_audio = False
        for ext in ["*.wav", "*.mp3", "*.flac"]:
            if next(pp.rglob(ext), None) is not None:
                has_audio = True
                break
        if has_audio:
            valid.append(p)
    return valid

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )

augmentation_probabilities = {
    "SevenBandParametricEQ": 0.1,
    "TanhDistortion": 0.1,
    "PitchShift": 0.1,
    "BandStopFilter": 0.1,
    "AddColorNoise": 0.1,
    "AddBackgroundNoise": 0.75,
    "Gain": 1.0,
    "RIR": 0.5,
}

background_paths = dirs_with_audio(['fma_16k', 'audioset_16k'])
if not background_paths:
    print("Warning: no background audio found. Disabling AddBackgroundNoise.")
    augmentation_probabilities["AddBackgroundNoise"] = 0.0

impulse_paths = dirs_with_audio(['mit_rirs'])
if not impulse_paths:
    print("Warning: no RIR files found. Disabling RIR augmentation.")
    augmentation_probabilities["RIR"] = 0.0

augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities=augmentation_probabilities,
                         impulse_paths=impulse_paths,
                         background_paths=background_paths,
                         background_min_snr_db=-5,
                         background_max_snr_db=10,
                         min_gain_db=-18,
                         max_gain_db=3,
                         min_jitter_s=0.195,
                         max_jitter_s=0.205,
                         )


In [ ]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio, display
from microwakeword.audio.audio_utils import save_clip
import numpy as np

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

peak = float(np.max(np.abs(augmented_clip)))
rms = float(np.sqrt(np.mean(np.square(augmented_clip))))
print(f"Raw augmented clip peak={peak:.6f}, rms={rms:.6f}")
print("Note: augmentation pads clips to 3.2s, so wakeword can be near the end.")

# Preview-only normalization for easier listening in notebook.
# Training data is still generated from raw augmentation values.
preview_clip = augmented_clip.astype(np.float32)
preview_peak = float(np.max(np.abs(preview_clip)))
if preview_peak > 0:
    preview_clip = preview_clip / preview_peak * 0.8
save_clip(preview_clip, 'augmented_clip_preview.wav')
display(Audio(filename="augmented_clip_preview.wav", autoplay=True))

In [ ]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

In [ ]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

import os
import zipfile
import urllib.request
from pathlib import Path

output_dir = './negative_datasets'
os.makedirs(output_dir, exist_ok=True)

link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']

for fname in filenames:
    link = link_root + fname
    zip_path = Path(output_dir) / fname
    if not zip_path.exists():
        print(f'Downloading {link} -> {zip_path}')
        urllib.request.urlretrieve(link, str(zip_path))
    else:
        print(f'Skip existing zip: {zip_path}')

    extract_dir = Path(output_dir) / fname.replace('.zip', '')
    if extract_dir.exists() and any(extract_dir.iterdir()):
        print(f'Skip extract (already exists): {extract_dir}')
        continue

    print(f'Extracting {zip_path} -> {output_dir}')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(output_dir)


In [ ]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    { # Only used for validation and testing
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
config["batch_size"] = 128

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    500  # Test the validation sets after every this many steps
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)

In [ ]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

import importlib.util
import sys

if importlib.util.find_spec("tensorboard") is None:
    !{sys.executable} -m pip install tensorboard

!{sys.executable} -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# Downloads the tflite model file. To use on the device, you need to write a
# Model JSON file. See https://esphome.io/components/micro_wake_word for the
# documentation and
# https://github.com/esphome/micro-wake-word-models/tree/main/models/v2 for
# examples. Adjust the probability threshold based on the test results obtained
# after training is finished. You may also need to increase the Tensor arena
# model size if the model fails to load.

from pathlib import Path
from IPython.display import FileLink, display

candidates = [
    Path("trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"),
    Path("microWakeWord/notebooks/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"),
]
found = [p.resolve() for p in candidates if p.exists()]
if not found:
    found = [p.resolve() for p in Path(".").rglob("stream_state_internal_quant.tflite")]

if not found:
    print("Model file not found yet. Run Cell 10 to completion first.")
    print("Expected path: trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")
else:
    model_path = found[0]
    print(f"Model file: {model_path}")
    print("Local environment detected. Use path/link below.")
    display(FileLink(str(model_path)))

In [ ]:
# Performance test on local recordings + visualization
# Default input dir:
#   /Volumes/Gold-P31-SSD-2TB/wakeword/nubjuga
# Fallback:
#   /Volumes/Gold-P31-SSD-2TB/wakeword/computer

from pathlib import Path
import math
import warnings

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from numpy.lib.stride_tricks import sliding_window_view
from scipy.io import wavfile

from microwakeword.inference import Model

recordings_dir_candidates = [
    Path("/Volumes/Gold-P31-SSD-2TB/wakeword/nubjuga"),
    Path("/Volumes/Gold-P31-SSD-2TB/wakeword/computer"),
]
recordings_dir = next((d for d in recordings_dir_candidates if d.exists()), recordings_dir_candidates[0])
cutoff = 0.78  # 큰소리 오탐이 많으면 0.80~0.90까지 올려서 테스트
ma_window = 4
step_ms = 10
cooldown_ms = 750
target_faph = 0.5

model_candidates = [
    Path("trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"),
    Path("microWakeWord/notebooks/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"),
    Path("microWakeWord/notebooks/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"),
]
model_path = next((p for p in model_candidates if p.exists()), None)
if model_path is None:
    found = sorted(Path(".").rglob("stream_state_internal_quant.tflite"))
    assert found, "Model file not found. Run Cell 10 (training) first."
    model_path = found[0]

stride = 3
training_cfg = model_path.parent.parent / "training_config.yaml"
if training_cfg.exists():
    try:
        cfg = yaml.safe_load(training_cfg.read_text()) or {}
        stride = int((cfg.get("flags", {}) or {}).get("stride", cfg.get("stride", stride)))
    except Exception:
        pass

assert recordings_dir.exists(), f"recordings dir not found: {recordings_dir}"

def moving_average(values, window):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values
    if values.size < window:
        return values
    return sliding_window_view(values, window).mean(axis=1)

def load_pcm16(path):
    if path.suffix.lower() == ".wav":
        sr, data = wavfile.read(path)
        if data.ndim > 1:
            data = data[:, 0]
        if np.issubdtype(data.dtype, np.floating):
            data = np.clip(data, -1.0, 1.0)
            data = (data * 32767).astype(np.int16)
        elif data.dtype != np.int16:
            data = data.astype(np.int16)
        if sr != 16000:
            audio = data.astype(np.float32) / 32768.0
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            data = np.clip(audio, -1.0, 1.0)
            data = (data * 32767).astype(np.int16)
        return data

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        audio, _ = librosa.load(str(path), sr=16000, mono=True)
    audio = np.clip(audio, -1.0, 1.0)
    return (audio * 32767).astype(np.int16)

def score_clip(path):
    pcm = load_pcm16(path)
    # Reset state by creating a fresh interpreter per file for fair per-clip eval.
    model = Model(str(model_path), stride=stride)
    probs = np.asarray(model.predict_clip(pcm, step_ms=step_ms), dtype=np.float32)
    smoothed = moving_average(probs, ma_window)
    return {
        "file": path.name,
        "frames": int(probs.size),
        "ma_max": float(np.max(smoothed)) if smoothed.size else 0.0,
        "smoothed": smoothed,
        "duration_h": len(pcm) / 16000.0 / 3600.0,
    }

def count_events(smoothed, threshold, cooldown_frames):
    events = 0
    cooldown = 0
    for s in smoothed:
        if cooldown > 0:
            cooldown -= 1
        if cooldown == 0 and s >= threshold:
            events += 1
            cooldown = cooldown_frames
    return events

positive_files = sorted(
    [p for p in recordings_dir.glob("*") if p.suffix.lower() in {".wav", ".m4a", ".mp3", ".flac", ".ogg"}]
)
assert positive_files, f"No audio files found in {recordings_dir}"

positive_results = [score_clip(p) for p in positive_files]
df_pos = pd.DataFrame([
    {
        "file": r["file"],
        "frames": r["frames"],
        "ma_max": r["ma_max"],
        f"detect@{cutoff:.2f}": r["ma_max"] >= cutoff,
    }
    for r in positive_results
]).sort_values("ma_max", ascending=False).reset_index(drop=True)

print(f"model={model_path}")
print(f"stride={stride} step_ms={step_ms} cutoff={cutoff:.2f}")
display(df_pos)
print(f"detection_rate@{cutoff:.2f}: {(df_pos[f'detect@{cutoff:.2f}'].mean()*100):.1f}%")

fig, ax = plt.subplots(figsize=(12, 4.8))
x = np.arange(len(df_pos))
ax.bar(x, df_pos["ma_max"].values, color="tab:blue", alpha=0.85)
ax.axhline(cutoff, color="tab:red", linestyle="--", linewidth=1.4, label=f"cutoff={cutoff:.2f}")
ax.set_xticks(x)
ax.set_xticklabels(df_pos["file"].values, rotation=35, ha="right")
ax.set_ylim(0.0, 1.05)
ax.set_ylabel("max moving-average score")
ax.set_title("Per-recording score")
ax.grid(alpha=0.25)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

n = len(positive_results)
ncols = 2
nrows = math.ceil(n / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.0*nrows), sharey=True)
axes = np.array(axes).reshape(-1)
for i, r in enumerate(sorted(positive_results, key=lambda x: x["file"])):
    sm = r["smoothed"]
    axes[i].plot(sm, color="tab:blue", linewidth=1.4)
    axes[i].axhline(cutoff, color="tab:red", linestyle="--", linewidth=1.0)
    axes[i].set_title(f"{r['file']} (max={r['ma_max']:.3f})", fontsize=10)
    axes[i].set_xlabel("frame")
    axes[i].set_ylim(0.0, 1.05)
    axes[i].grid(alpha=0.25)
for j in range(i + 1, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

negative_dir_candidates = [Path("fma_16k"), Path("microWakeWord/notebooks/fma_16k")]
negative_dir = next((d for d in negative_dir_candidates if d.exists()), None)
if negative_dir is None:
    print("No negative dataset directory found; skipped threshold tradeoff graph.")
else:
    negative_files = sorted(negative_dir.glob("*.wav"))
    print(f"negative_set={negative_dir} count={len(negative_files)}")
    if len(negative_files) > 0:
        negative_results = [score_clip(p) for p in negative_files]
        pos_scores = np.asarray([r["ma_max"] for r in positive_results], dtype=np.float32)
        thresholds = np.arange(0.0, 1.0001, 0.01, dtype=np.float32)
        cooldown_frames = max(0, int(round(cooldown_ms / step_ms)))
        total_negative_hours = float(sum(r["duration_h"] for r in negative_results))

        frrs = []
        faphs = []
        for th in thresholds:
            frrs.append(1.0 - float(np.mean(pos_scores >= th)))
            fp_events = 0
            for r in negative_results:
                fp_events += count_events(r["smoothed"], float(th), cooldown_frames)
            faphs.append(fp_events / max(total_negative_hours, 1e-12))

        frrs = np.asarray(frrs, dtype=np.float32)
        faphs = np.asarray(faphs, dtype=np.float32)
        feasible = np.where(faphs <= target_faph)[0]
        if feasible.size > 0:
            best_idx = int(feasible[np.argmin(frrs[feasible])])
        else:
            penalty = np.maximum(0.0, faphs - target_faph) * 10.0 + frrs
            best_idx = int(np.argmin(penalty))
        cutoff_idx = int(np.argmin(np.abs(thresholds - cutoff)))
        print(f"cutoff={cutoff:.2f}: FRR={frrs[cutoff_idx]:.4f}, FAPH={faphs[cutoff_idx]:.4f}")
        print(f"recommended_threshold={thresholds[best_idx]:.2f}: FRR={frrs[best_idx]:.4f}, FAPH={faphs[best_idx]:.4f}")

        fig, ax1 = plt.subplots(figsize=(10, 5.2))
        ax1.plot(thresholds, frrs, color="tab:blue", linewidth=2.0, label="FRR")
        ax1.set_xlabel("threshold")
        ax1.set_ylabel("FRR", color="tab:blue")
        ax1.tick_params(axis="y", labelcolor="tab:blue")
        ax1.grid(alpha=0.25)

        ax2 = ax1.twinx()
        ax2.plot(thresholds, faphs, color="tab:red", linewidth=2.0, label="FAPH")
        ax2.set_ylabel("FAPH (/hour)", color="tab:red")
        ax2.tick_params(axis="y", labelcolor="tab:red")

        ax1.axvline(cutoff, color="tab:green", linestyle="--", linewidth=1.3, label=f"cutoff={cutoff:.2f}")
        ax1.axvline(float(thresholds[best_idx]), color="tab:purple", linestyle=":", linewidth=1.5, label=f"recommended={thresholds[best_idx]:.2f}")

        l1, lb1 = ax1.get_legend_handles_labels()
        l2, lb2 = ax2.get_legend_handles_labels()
        ax1.legend(l1 + l2, lb1 + lb2, loc="upper right")
        ax1.set_title("Threshold tradeoff (your recordings vs negative set)")
        plt.tight_layout()
        plt.show()

In [ ]:
# Realtime microphone wakeword test (local)
# Model path requested:
#   trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite
#
# 사용 순서:
# 1) 이 셀 실행 -> 입력 장치 목록 확인
# 2) selected_device 를 목록의 입력 장치 번호로 설정
# 3) RUN_REALTIME = True 후 셀 재실행

import sys
import subprocess
from pathlib import Path

model_candidates = [
    Path("trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"),
    Path("microWakeWord/notebooks/trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"),
]
model_path = next((p for p in model_candidates if p.exists()), None)
if model_path is None:
    found = sorted(Path(".").rglob("stream_state_internal_quant.tflite"))
    assert found, "Model file not found. Run training cell first."
    model_path = found[0]
script_candidates = [
    Path("../../../scripts/09_realtime_mic_test.py"),
    Path("scripts/09_realtime_mic_test.py"),
    Path("/Users/unripeplum/projects/nubjuk/wakeword/scripts/09_realtime_mic_test.py"),
]
script_path = next((p for p in script_candidates if p.exists()), None)
assert script_path is not None, "09_realtime_mic_test.py not found"

print(f"script={script_path.resolve()}")
print(f"model={model_path.resolve()}")
print("\n[1/2] Input devices:")
subprocess.run([sys.executable, str(script_path), "--list-devices"], check=False)

selected_device = None  # 예: 4
cutoff = 0.78  # 큰소리 오탐이 많으면 0.80~0.90까지 올려서 테스트
preamp = 1.5
score_mode = "rolling_window"  # 권장: block 경계 영향 완화
rolling_window_ms = 1500
rolling_min_ms = 300
block_ms = 200
trigger_hold_blocks = 2
rearm_ratio = 0.6
rearm_hold_blocks = 2
rearm_threshold = None  # None이면 cutoff * rearm_ratio 사용
RUN_REALTIME = False

cmd = [
    sys.executable,
    str(script_path),
    "--model",
    str(model_path),
    "--cutoff",
    str(cutoff),
    "--preamp",
    str(preamp),
    "--score-mode",
    str(score_mode),
    "--rolling-window-ms",
    str(rolling_window_ms),
    "--rolling-min-ms",
    str(rolling_min_ms),
    "--block-ms",
    str(block_ms),
    "--trigger-hold-blocks",
    str(trigger_hold_blocks),
    "--rearm-ratio",
    str(rearm_ratio),
    "--rearm-hold-blocks",
    str(rearm_hold_blocks),
]
if rearm_threshold is not None:
    cmd += ["--rearm-threshold", str(rearm_threshold)]
if selected_device is not None:
    cmd += ["--device", str(selected_device)]

print("\n[2/2] Realtime command:")
print(" ".join(cmd))
if RUN_REALTIME:
    print("Starting realtime test... stop with Ctrl+C or Kernel Interrupt")
    subprocess.run(cmd, check=False)
else:
    print("RUN_REALTIME=False 이므로 실행하지 않았습니다. 값을 True로 바꿔 재실행하세요.")